<a href="https://colab.research.google.com/github/angelrecalde2024/Power-System-Planning-and-Transmission-Design-2026/blob/main/INGP1118_Sag_tension_OHL_table.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

MECHANICAL DESIGN VERIFICATION

This script performs the mechanical dsign verification of an overhead transmission conductor by computing:

*  Mechanical tension $T_2$ under several environmental conditions
*  Sag (flecha) for different spans (vanos)
*  A summary report table

The calculations are based on the state equation of a conductor between two mechanical states:

1.  State 1: reference condition (EDS - everyday stress)
2.  State 2: environmental load condition (temperature + wind)

The output is evaluated for multiple span lengths.

PHYSICAL MODEL IMPLEMENTED

The conductor is modelled using the catenary approximation. Sag is computed using the catenary parameter:

$$
P = \frac{T}{\omega}
$$

The sag formula is:

$$
f=P\left(\cosh \left(\frac{V}{2 P}\right)-1\right)
$$

where $V$ is the span, $T$ is the horizontal tension, and $\omega$ is the conductor weight per meter.

The conductor mechanical properties are introduced via an Excel spreadsheet file with the following properties:

*  breaking strengh
*  weight per length
*  diameter
*  elastic modulus
*  area
*  thermal expansion

These define the mechanical behavior of the conductor.

Wind pressure is modeled as

$$
w_p=K v^2
$$

where $K$ is the wind pressure constant, and $v$ is the wind speed. The wind produces a horizontal load component whose equivalent load is:

$$
w_w=w_p D
$$

The total load in the conductor due to weight and wind is:

$$
w_2=\sqrt{w_1^2+w_w^2}
$$

The tension in state 2 is obtained from a cubic equation:

$$
x^3+K_1 x^2-K_2=0
$$

where

$$
\begin{gathered}
K_1=\left(Y A \alpha\left(t_2-t_1\right)\right)-T_1+\frac{V^2 w_1^2 Y A}{24 T_1^2} \\
K_2=\frac{V^2 w_2^2 Y A}{24}
\end{gathered}
$$

The unknown variable is $x=T_2$. It is solved using numerical root finding.

DATA REQUIRED

The mechanical study is performed for a set of spans in meters. Each span is analyzed independently. There must be an Excel book with the following information:

* 'Format': this is the report format. The output will copy the report and replace the values accordingly.
*  'ConductorData': this sheet contains the conductor data for every conductor that will be used in the overhead line. It must containt: type, caliber, Al/St composition, weight per meter, breaking strength, diameter, total area, elastic modulus, thermal expansion.
*  'spans': sheet with all the span values to be calculated. It has the header 'span (m)'.
*  'states': sheet with the base state and the state conditions to be calculated. It has: EDS (everyday stress), minimum temperatura (state 1), maximum load (state 2), maximum temperature (state 3).

OUTPUT

The results are exported to an Excel workbook that must be downloaded.





In [3]:
# ============================================
# 1) Imports
# ============================================
import math
import re
from dataclasses import dataclass
from typing import Dict, List, Tuple

import numpy as np
import openpyxl
from openpyxl.worksheet.worksheet import Worksheet
from openpyxl.styles import Font, PatternFill, Border, Alignment, Protection, Side
from openpyxl.utils import get_column_letter
from openpyxl.chart import ScatterChart, Reference, Series

from scipy.optimize import fsolve

# Constante de presión de viento (hardcoded según tu instrucción)
K_WIND = 0.0048

OK_FILL = PatternFill(start_color="C6EFCE", end_color="C6EFCE", fill_type="solid")
NOK_FILL = PatternFill(start_color="FFC7CE", end_color="FFC7CE", fill_type="solid")

# ============================================
# 2) Utilidades: parsing robusto (números / placeholders)
# ============================================

PLACEHOLDER_RE = re.compile(r"^<[^>]+>$")

def is_placeholder(x) -> bool:
    return isinstance(x, str) and PLACEHOLDER_RE.match(x.strip()) is not None

def to_float(x, field_name: str) -> float:
    """
    Convierte a float soportando:
    - números ya numéricos
    - strings con coma decimal "1,23"
    - strings con espacios
    Rechaza placeholders tipo "<...>"
    """
    if x is None:
        raise ValueError(f"Campo '{field_name}' está vacío (None).")
    if is_placeholder(x):
        raise ValueError(f"Campo '{field_name}' sigue siendo placeholder: {x}")
    if isinstance(x, (int, float)):
        return float(x)
    if isinstance(x, str):
        s = x.strip().replace(" ", "")
        s = s.replace(",", ".")
        try:
            return float(s)
        except:
            raise ValueError(f"No se pudo convertir '{field_name}'='{x}' a número.")
    raise ValueError(f"Tipo no soportado para '{field_name}': {type(x)}")

def to_str(x, field_name: str) -> str:
    if x is None:
        raise ValueError(f"Campo '{field_name}' está vacío (None).")
    if is_placeholder(x):
        raise ValueError(f"Campo '{field_name}' sigue siendo placeholder: {x}")
    return str(x).strip()

def pct_to_frac(pct_value: float) -> float:
    """
    En tu plantilla, la tensión límite está en %.
    Si alguien pone 0.14, también lo aceptamos como fracción.
    Regla:
      - si > 1.5 => interpretamos como porcentaje (14 => 0.14)
      - si <= 1.5 => interpretamos como fracción (0.14 => 0.14)
    """
    if pct_value > 1.5:
        return pct_value / 100.0
    return pct_value

def add_charts(ws, start_row, n_spans):

    last_row = start_row + n_spans - 1

    # -----------------------------
    # Gráfico 1: Flecha vs Vano
    # -----------------------------
    chart1 = ScatterChart()
    chart1.title = "Flecha vs Vano"
    chart1.x_axis.title = "Vano (m)"
    chart1.y_axis.title = "Flecha (m)"

    xvalues = Reference(ws, min_col=2, min_row=start_row, max_row=last_row)
    yvalues = Reference(ws, min_col=9, min_row=start_row, max_row=last_row)

    series = Series(yvalues, xvalues, title="Flecha máxima")
    chart1.series.append(series)

    ws.add_chart(chart1, "K5")

    # -----------------------------
    # Gráfico 2: Tensión vs Vano
    # -----------------------------
    chart2 = ScatterChart()
    chart2.title = "Tensión vs Vano"
    chart2.x_axis.title = "Vano (m)"
    chart2.y_axis.title = "Tensión (kgf)"

    t1 = Reference(ws, min_col=3, min_row=start_row, max_row=last_row)
    t2 = Reference(ws, min_col=5, min_row=start_row, max_row=last_row)
    t3 = Reference(ws, min_col=7, min_row=start_row, max_row=last_row)

    chart2.series.append(Series(t1, xvalues, title="Estado 1"))
    chart2.series.append(Series(t2, xvalues, title="Estado 2"))
    chart2.series.append(Series(t3, xvalues, title="Estado 3"))

    ws.add_chart(chart2, "K20")

    # ============================================
# 3) Lectura del Excel de entrada
# ============================================

@dataclass
class Conductor:
    tipo: str
    calibre: str
    composicion: str
    wm: float  # kg/m
    Tr: float  # kgf
    Dm: float  # mm
    At: float  # mm2
    Ym: float  # kg/mm2
    Cd: float  # 1/C

@dataclass
class States:
    # EDS base:
    t0: float
    lim0: float  # fracción (0.14) ya convertida
    v0: float

    # Estados 1..3:
    t1: float; lim1: float; v1: float
    t2: float; lim2: float; v2: float
    t3: float; lim3: float; v3: float

def read_conductor_data(wb: openpyxl.Workbook) -> Tuple[Conductor, Conductor]:
    ws = wb["ConductorData"]

    # Filas fijas (según tu plantilla actual)
    # 2: tipo, 3: calibre, 4: composición, 5..10 numéricos
    def col(c):  # c=2 conductor1, c=3 conductor2
        return {
            "tipo": to_str(ws.cell(2, c).value, "tipo"),
            "calibre": to_str(ws.cell(3, c).value, "calibre"),
            "composicion": to_str(ws.cell(4, c).value, "composicion"),
            "wm": to_float(ws.cell(5, c).value, "pesoMetro"),
            "Tr": to_float(ws.cell(6, c).value, "tensionRotura"),
            "Dm": to_float(ws.cell(7, c).value, "diametro"),
            "At": to_float(ws.cell(8, c).value, "areaTotal"),
            "Ym": to_float(ws.cell(9, c).value, "Elasticidad"),
            "Cd": to_float(ws.cell(10, c).value, "dilatacion"),
        }

    c1 = col(2)
    c2 = col(3)

    return (
        Conductor(**c1),
        Conductor(**c2),
    )

def read_spans(wb: openpyxl.Workbook) -> List[float]:
    ws = wb["spans"]
    spans = []
    for r in range(2, ws.max_row + 1):
        v = ws.cell(r, 1).value
        if v is None:
            continue
        spans.append(to_float(v, f"span_row_{r}"))
    if not spans:
        raise ValueError("No se encontraron vanos en la hoja 'spans'.")
    return spans

def read_states_for_conductor(ws_states: Worksheet, start_row: int) -> States:
    """
    En tu plantilla:
      Conductor 1 empieza en fila 2 (bloque ocupa 2..5)
      Conductor 2 empieza en fila 8 (bloque ocupa 8..11)
    Filas relativas:
      start_row+1: Temperatura
      start_row+2: Tensión límite (%)
      start_row+3: Viento
    Columnas:
      col2=EDS, col3=Estado1, col4=Estado2, col5=Estado3
    """
    # Temperaturas
    T = [ws_states.cell(start_row + 1, c).value for c in range(2, 6)]
    # Límites
    L = [ws_states.cell(start_row + 2, c).value for c in range(2, 6)]
    # Vientos
    V = [ws_states.cell(start_row + 3, c).value for c in range(2, 6)]

    T = [to_float(x, f"temperatura_{i}") for i, x in enumerate(T)]
    L = [pct_to_frac(to_float(x, f"tensionLim_{i}")) for i, x in enumerate(L)]
    V = [to_float(x, f"viento_{i}") for i, x in enumerate(V)]

    return States(
        t0=T[0], lim0=L[0], v0=V[0],
        t1=T[1], lim1=L[1], v1=V[1],
        t2=T[2], lim2=L[2], v2=V[2],
        t3=T[3], lim3=L[3], v3=V[3],
    )

def read_states(wb: openpyxl.Workbook) -> Tuple[States, States]:
    ws = wb["states"]
    st1 = read_states_for_conductor(ws, start_row=2)  # Conductor 1 bloque
    st2 = read_states_for_conductor(ws, start_row=8)  # Conductor 2 bloque
    return st1, st2

# ============================================
# 4) Modelo mecánico (paridad con MATLAB)
# ============================================

def total_weight(w1: float, vw_kmh: float, Dm_mm: float) -> float:
    """
    MATLAB:
      wp = K*(vw2^2)
      ww = wp*(Dm*(1e-3))
      w2 = sqrt(w1^2 + ww^2)
    """
    wp = K_WIND * (vw_kmh ** 2)
    D_m = Dm_mm * 1e-3
    ww = wp * D_m
    return math.sqrt((w1 ** 2) + (ww ** 2))

def solve_state_eq(Ym: float, At: float, Cd: float,
                   t1: float, t2: float,
                   T1: float, Vspan: float,
                   w1: float, w2: float,
                   x0: float) -> float:
    """
    MATLAB:
      K1 = (Ym*At*Cd*(t2-t1)) - T1 + ((V^2*w1^2*Ym*At)/(24*T1^2))
      K2 = ((V^2)*(w2^2)*Ym*At)/24
      stateEQ(x) = x^3 + K1*x^2 - K2 = 0
      T2 = fsolve(stateEQ, x0)
    """
    K1 = (Ym * At * Cd * (t2 - t1)) - T1 + (((Vspan ** 2) * (w1 ** 2) * Ym * At) / (24.0 * (T1 ** 2)))
    K2 = (((Vspan ** 2) * (w2 ** 2) * Ym * At) / 24.0)

    def f(x):
        return (x ** 3) + (K1 * (x ** 2)) - K2

    sol = fsolve(f, x0=x0, xtol=1e-9, maxfev=500)
    T2 = float(sol[0])
    return T2

def sag_catenary(T: float, w: float, Vspan: float) -> float:
    """
    MATLAB:
      P = T2/w2
      fl = P*(cosh(V/(2P))-1)
    """
    P = T / w
    return P * (math.cosh(Vspan / (2.0 * P)) - 1.0)

 # ============================================
# 5) Cálculo por conductor (genera tabla tipo RTable)
# ============================================

def run_conductor_case(conductor: Conductor, states: States, spans: List[float]) -> Dict:
    """
    Devuelve un dict con:
      - T_EDS
      - results: lista de filas por vano con tensiones y flechas
    """
    # Estado base (EDS)
    T1 = states.lim0 * conductor.Tr
    w1 = conductor.wm
    t_base = states.t0

    rows = []
    for Vspan in spans:
        # Estado 1
        w2_1 = total_weight(w1, states.v1, conductor.Dm)
        T2_1 = solve_state_eq(conductor.Ym, conductor.At, conductor.Cd,
                              t_base, states.t1,
                              T1, Vspan, w1, w2_1,
                              x0=1.25 * T1)
        f1 = sag_catenary(T2_1, w2_1, Vspan)

        # Estado 2
        w2_2 = total_weight(w1, states.v2, conductor.Dm)
        T2_2 = solve_state_eq(conductor.Ym, conductor.At, conductor.Cd,
                              t_base, states.t2,
                              T1, Vspan, w1, w2_2,
                              x0=1.25 * T1)
        f2 = sag_catenary(T2_2, w2_2, Vspan)

        # Estado 3
        w2_3 = total_weight(w1, states.v3, conductor.Dm)
        T2_3 = solve_state_eq(conductor.Ym, conductor.At, conductor.Cd,
                              t_base, states.t3,
                              T1, Vspan, w1, w2_3,
                              x0=1.25 * T1)
        f3 = sag_catenary(T2_3, w2_3, Vspan)

        fmax = max(f1, f2, f3)

        # Chequeos vs límites (si deseas reportarlos luego)
        ok1 = T2_1 <= (states.lim1 * conductor.Tr)
        ok2 = T2_2 <= (states.lim2 * conductor.Tr)
        ok3 = T2_3 <= (states.lim3 * conductor.Tr)

        verdict1 = "OK" if ok1 else "NO OK"
        verdict2 = "OK" if ok2 else "NO OK"
        verdict3 = "OK" if ok3 else "NO OK"

        rows.append({
            "span_m": Vspan,

            "T_E1": T2_1,
            "verdict1": verdict1,

            "T_E2": T2_2,
            "verdict2": verdict2,

            "T_E3": T2_3,
            "verdict3": verdict3,

            "f_max": fmax
        })

    return {
        "T_EDS": T1,
        "rows": rows
    }

# ============================================
# 6) Escritura del Excel de salida con el formato del template
# ============================================

from copy import copy

def copy_cell_style(src_cell, dst_cell):

    if src_cell.has_style:

        dst_cell.font = copy(src_cell.font)
        dst_cell.border = copy(src_cell.border)
        dst_cell.fill = copy(src_cell.fill)
        dst_cell.number_format = copy(src_cell.number_format)
        dst_cell.protection = copy(src_cell.protection)
        dst_cell.alignment = copy(src_cell.alignment)

def replace_placeholders_in_sheet(ws: Worksheet, mapping: Dict[str, object]):
    """
    Reemplaza cualquier celda cuyo valor sea exactamente "<...>" y esté en mapping.
    """
    for row in ws.iter_rows():
        for cell in row:
            v = cell.value
            if isinstance(v, str) and v.strip() in mapping:
                cell.value = mapping[v.strip()]

def ensure_rows_for_spans(ws: Worksheet, start_row: int, template_row: int, n_rows: int):
    """
    Asegura que existen n_rows filas a partir de start_row, copiando formato desde template_row.
    Si la plantilla tiene 10 y necesitas más, inserta y copia estilos.
    """
    existing = ws.max_row
    needed_last = start_row + n_rows - 1
    if needed_last <= existing:
        return

    # Insertar las filas faltantes al final del bloque
    rows_to_add = needed_last - existing
    ws.insert_rows(existing + 1, amount=rows_to_add)

    # Copiar formato desde template_row hacia las nuevas filas insertadas
    for r in range(existing + 1, needed_last + 1):
        for c in range(1, ws.max_column + 1):
            src = ws.cell(template_row, c)
            dst = ws.cell(r, c)
            copy_cell_style(src, dst)

def write_results_table(ws, spans_rows, start_row):

    for i, rr in enumerate(spans_rows):

        r = start_row + i

        ws.cell(r,2).value = rr["span_m"]

        ws.cell(r,3).value = rr["T_E1"]
        ws.cell(r,4).value = rr["verdict1"]

        ws.cell(r,5).value = rr["T_E2"]
        ws.cell(r,6).value = rr["verdict2"]

        ws.cell(r,7).value = rr["T_E3"]
        ws.cell(r,8).value = rr["verdict3"]

        ws.cell(r,9).value = rr["f_max"]

        # colorear veredictos
        if rr["verdict1"] == "OK":
            ws.cell(r,4).fill = OK_FILL
        else:
            ws.cell(r,4).fill = NOK_FILL

        if rr["verdict2"] == "OK":
            ws.cell(r,6).fill = OK_FILL
        else:
            ws.cell(r,6).fill = NOK_FILL

        if rr["verdict3"] == "OK":
            ws.cell(r,8).fill = OK_FILL
        else:
            ws.cell(r,8).fill = NOK_FILL

def build_mapping(conductor: Conductor, states: States, T_EDS: float) -> Dict[str, object]:
    """
    Mapea placeholders del sheet Format a valores reales.
    """
    return {
        "<tipo>": conductor.tipo,
        "<calibre>": conductor.calibre,
        "<composicion>": conductor.composicion,
        "<pesoMetro>": conductor.wm,
        "<tensionRotura>": conductor.Tr,
        "<diametro>": conductor.Dm,
        "<areaTotal>": conductor.At,
        "<Elasticidad>": conductor.Ym,
        "<dilatacion>": conductor.Cd,

        "<temperatura0>": states.t0,
        "<temperatura1>": states.t1,
        "<temperatura2>": states.t2,
        "<temperatura3>": states.t3,

        "<tensionLim0>": states.lim0 * 100.0,
        "<tensionLim1>": states.lim1 * 100.0,
        "<tensionLim2>": states.lim2 * 100.0,
        "<tensionLim3>": states.lim3 * 100.0,

        "<viento0>": states.v0,
        "<viento1>": states.v1,
        "<viento2>": states.v2,
        "<viento3>": states.v3,

        "<tensionEDSv>": T_EDS,
    }

def generate_output_excel(template_path: str, output_path: str):
    wb = openpyxl.load_workbook(template_path)

    # Leer entradas
    c1, c2 = read_conductor_data(wb)
    spans = read_spans(wb)
    st1, st2 = read_states(wb)

    # Cálculos
    res1 = run_conductor_case(c1, st1, spans)
    res2 = run_conductor_case(c2, st2, spans)

    # Preparar libro de salida (copiar Format dos veces)
    ws_format = wb["Format"]
    ws1 = wb.copy_worksheet(ws_format)
    ws2 = wb.copy_worksheet(ws_format)
    ws1.title = "Conductor_1"
    ws2.title = "Conductor_2"

    # (Opcional) mantener el template original o borrarlo
    # wb.remove(ws_format)

    # Rellenar placeholders
    replace_placeholders_in_sheet(ws1, build_mapping(c1, st1, res1["T_EDS"]))
    replace_placeholders_in_sheet(ws2, build_mapping(c2, st2, res2["T_EDS"]))

    # Escribir tabla de vanos: empieza en fila 22, usar fila 22 como fila-template
    START_ROW = 22
    TEMPLATE_ROW = 22
    ensure_rows_for_spans(ws1, START_ROW, TEMPLATE_ROW, n_rows=len(spans))
    ensure_rows_for_spans(ws2, START_ROW, TEMPLATE_ROW, n_rows=len(spans))

    write_results_table(ws1, res1["rows"], START_ROW)
    write_results_table(ws2, res2["rows"], START_ROW)

    add_charts(ws1, START_ROW, len(spans))
    add_charts(ws2, START_ROW, len(spans))

    wb.save(output_path)
    return output_path

# ============================================
# 7) Ejecutar
# ============================================

TEMPLATE_XLSX = "/content/Formato_Calculo_tensiones_mec_lineas_v1.xlsx"  # ajusta si el nombre cambia
OUTPUT_XLSX = "/content/Resultados_Calculo_Tensiones.xlsx"

out = generate_output_excel(TEMPLATE_XLSX, OUTPUT_XLSX)
print("Archivo generado:", out)

Archivo generado: /content/Resultados_Calculo_Tensiones.xlsx


In [ ]:
# En Colab, para descargar:
from google.colab import files
files.download(OUTPUT_XLSX)